# Atividade 9 - Entrega Final (Encontro 10)

Projeto: recomendacao de treinos personalizados em academia.

Esta e a versao final e revisada do esboco entregue no Encontro 9. Nesta etapa, alem do treinamento do baseline e do primeiro modelo, foram adicionadas: matriz de confusao, F1 por classe, interpretacao das regras aprendidas pela arvore e verificacao de vazamento de dados, a partir do feedback recebido sobre o esboco.

## 1. Base tratada carregada

A base foi carregada e as colunas foram inspecionadas antes do treinamento. A coluna `aluno_id` foi removida das features, porque ela funciona como identificador e nao ajuda o modelo a aprender o padrao de recomendacao.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
df = pd.read_csv('dataset_academia_tratado.csv')
print(df.head().to_string(index=False))
print('\nShape:', df.shape)
print('Colunas:', list(df.columns))

 aluno_id  idade        objetivo nivel_experiencia  dias_disponiveis_semana restricao_fisica equipamento_disponivel treino_recomendado
        1     32 Condicionamento          Avancado                        2          Nenhuma               Completo           Treino C
        2     37     Hipertrofia         Iniciante                        4          Nenhuma               Completo           Treino A
        3     24 Condicionamento         Iniciante                        3           Lombar               Limitado           Treino D
        4     31   Emagrecimento         Iniciante                        4           Joelho                 Nenhum           Treino D
        5     22 Condicionamento     Intermediario                        3          Nenhuma               Completo           Treino C

Shape: (300, 8)
Colunas: ['aluno_id', 'idade', 'objetivo', 'nivel_experiencia', 'dias_disponiveis_semana', 'restricao_fisica', 'equipamento_disponivel', 'treino_recomendado']


In [ ]:
X = df.drop(columns=['aluno_id', 'treino_recomendado'])
y = df['treino_recomendado']
print('X shape:', X.shape)
print('y shape:', y.shape)
print('Colunas de X:', list(X.columns))

X shape: (300, 6)
y shape: (300,)
Colunas de X: ['idade', 'objetivo', 'nivel_experiencia', 'dias_disponiveis_semana', 'restricao_fisica', 'equipamento_disponivel']


## 2. Expectativa inicial (antes do treinamento)

Antes de rodar o modelo, espero uma acuracia acima do baseline, mas sem expectativa exagerada. A base tem padroes claros entre perfil do aluno e treino recomendado, mas ainda existe ruido e algumas excecoes. Por isso, considero plausivel um desempenho em torno de 60% a 80% de acertos.

## 3. Divisao treino e teste

Criterio usado: 80% para treino e 20% para teste. A divisao foi feita com `random_state=42` e `stratify=y` para manter a proporcao das classes nos dois conjuntos.

In [ ]:
X = pd.get_dummies(X, columns=['objetivo', 'nivel_experiencia', 'restricao_fisica', 'equipamento_disponivel'], drop_first=False)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print('Treino X:', X_train.shape)
print('Teste X:', X_test.shape)
print('Treino y:', y_train.shape)
print('Teste y:', y_test.shape)

Treino X: (240, 15)
Teste X: (60, 15)
Treino y: (240,)
Teste y: (60,)


## 4. Baseline treinado e avaliado

O baseline usa a estrategia `most_frequent`. Ele representa a resposta mais comum da base e funciona como regua minima para verificar se o modelo real esta aprendendo algo util.

In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_score = baseline.score(X_test, y_test)
print('Acuracia do baseline:', round(baseline_score, 4))

Acuracia do baseline: 0.3


## 5. Modelo escolhido

Escolhi a Arvore de Decisao porque o problema e de classificacao multiclasse. A base tem poucas variaveis, a maioria categorica, e a arvore e interpretavel, facil de explicar e muito adequada para demonstrar visualmente como as decisoes sao tomadas.

In [ ]:
modelo = DecisionTreeClassifier(max_depth=5, random_state=42)
modelo.fit(X_train, y_train)
predicoes = modelo.predict(X_test)
acuracia_modelo = accuracy_score(y_test, predicoes)
print('Acuracia do modelo:', round(acuracia_modelo, 4))

Acuracia do modelo: 0.9


## 6. Previsoes sobre o conjunto de teste

Abaixo esta uma amostra das previsoes do modelo comparadas com os valores reais.

In [ ]:
amostra = pd.DataFrame({
    'real': y_test.reset_index(drop=True),
    'prevista': predicoes
})
print(amostra.head(10).to_string(index=False))

    real prevista
Treino B Treino B
Treino A Treino A
Treino A Treino A
Treino A Treino A
Treino A Treino A
Treino A Treino A
Treino C Treino C
Treino A Treino A
Treino D Treino D
Treino A Treino A


## 7. Comparacao com o baseline e com a expectativa inicial

O modelo real superou claramente o baseline. Isso indica que ele aprendeu padroes uteis da base e nao apenas repetiu a classe mais frequente. O resultado ficou acima do que eu esperava inicialmente, o que sugere que o perfil do aluno esta bem relacionado com a recomendacao de treino.

In [ ]:
print('Baseline:', round(baseline_score, 4))
print('Modelo:', round(acuracia_modelo, 4))
print('Diferenca:', round(acuracia_modelo - baseline_score, 4))
print('Modelo superou o baseline?', acuracia_modelo > baseline_score)

Baseline: 0.3
Modelo: 0.9
Diferenca: 0.6
Modelo superou o baseline? True


## 8. Matriz de confusao e F1 por classe

A acuracia agregada (0,90) pode esconder o desempenho de uma classe especifica, principalmente porque as classes nao tem o mesmo tamanho (Treino D: 92, Treino C: 80, Treino A: 74, Treino B: 54 no total da base). Por isso, a matriz de confusao e o F1 por classe foram adicionados nesta entrega final.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

labels_ordenados = sorted(y.unique())
cm = confusion_matrix(y_test, predicoes, labels=labels_ordenados)
cm_df = pd.DataFrame(
    cm,
    index=[f'real_{l}' for l in labels_ordenados],
    columns=[f'prev_{l}' for l in labels_ordenados]
)
print('Matriz de confusao (linhas = real, colunas = previsto):')
print(cm_df)
print()
print('Relatorio de classificacao (precisao, recall e F1 por classe):')
print(classification_report(y_test, predicoes, labels=labels_ordenados, digits=2))

Matriz de confusao (linhas = real, colunas = previsto):
               prev_Treino A  prev_Treino B  prev_Treino C  prev_Treino D
real_Treino A             15              0              0              0
real_Treino B              1             10              0              0
real_Treino C              1              0             15              0
real_Treino D              2              1              1             14

Relatorio de classificacao (precisao, recall e F1 por classe):
              precision    recall  f1-score   support

    Treino A       0.79      1.00      0.88        15
    Treino B       0.91      0.91      0.91        11
    Treino C       0.94      0.94      0.94        16
    Treino D       1.00      0.78      0.88        18

    accuracy                           0.90        60
   macro avg       0.91      0.91      0.90        60
weighted avg       0.91      0.90      0.90        60


## 9. Regras aprendidas pela arvore (interpretabilidade)

Como a Arvore de Decisao e interpretavel, e possivel extrair as regras que ela aprendeu em formato de texto e comparar com a logica usada para montar o problema, em vez de tratar o modelo como uma caixa-preta.

In [ ]:
from sklearn.tree import export_text

regras_arvore = export_text(modelo, feature_names=list(X.columns))
print(regras_arvore)

|--- objetivo_Emagrecimento <= 0.50
|   |--- objetivo_Hipertrofia <= 0.50
|   |   |--- restricao_fisica_Nenhuma <= 0.50
|   |   |   |--- nivel_experiencia_Iniciante <= 0.50
|   |   |   |   |--- restricao_fisica_Lombar <= 0.50
|   |   |   |   |   |--- class: Treino C
|   |   |   |   |--- restricao_fisica_Lombar >  0.50
|   |   |   |   |   |--- class: Treino D
|   |   |   |--- nivel_experiencia_Iniciante >  0.50
|   |   |   |   |--- dias_disponiveis_semana <= 2.50
|   |   |   |   |   |--- class: Treino A
|   |   |   |   |--- dias_disponiveis_semana >  2.50
|   |   |   |   |   |--- class: Treino D
|   |   |--- restricao_fisica_Nenhuma >  0.50
|   |   |   |--- equipamento_disponivel_Nenhum <= 0.50
|   |   |   |   |--- nivel_experiencia_Iniciante <= 0.50
|   |   |   |   |   |--- class: Treino C
|   |   |   |   |--- nivel_experiencia_Iniciante >  0.50
|   |   |   |   |   |--- class: Treino C
|   |   |   |--- equipamento_disponivel_Nenhum >  0.50
|   |   |   |   |--- class: Treino D
|   |--- 

## 10. Verificacao de vazamento e leitura critica do resultado

A acuracia de 0,90 ficou acima da faixa esperada (0,60 a 0,80), o que e motivo de desconfianca antes de comemorar. Duas checagens foram feitas:

**a) Vazamento de dados:** `aluno_id` foi removido das features antes do treino, e nenhuma das colunas restantes e calculada a partir do rotulo ou so existe depois da recomendacao ser definida. Nao ha vazamento no sentido classico.

**b) Por que o resultado ficou tao alto entao?** A base e sintetica: `treino_recomendado` foi gerado por uma funcao de regras (`recomenda_treino`, em `gerar_dataset.py`), com ~8% de ruido aleatorio aplicado de proposito para simular excecoes. Isso significa que existe um teto esperado de desempenho: mesmo um modelo que recuperasse as regras perfeitamente erraria os ~8% de registros ruidosos. A celula abaixo recalcula, para cada aluno, qual seria a recomendacao "pura" da regra (sem ruido) e compara com o rotulo real e com a previsao do modelo.

In [ ]:
import sys
sys.path.append('.')
from gerar_dataset import recomenda_treino

df_regra = df.copy()
df_regra['regra_pura'] = df_regra.apply(
    lambda r: recomenda_treino(
        r['objetivo'], r['nivel_experiencia'], r['dias_disponiveis_semana'],
        r['restricao_fisica'], r['equipamento_disponivel']
    ),
    axis=1
)
df_regra['eh_ruido'] = df_regra['regra_pura'] != df_regra['treino_recomendado']
print('Registros com ruido de rotulo na base toda (regra pura != rotulo real):',
      df_regra['eh_ruido'].sum(), 'de', len(df_regra),
      f"({df_regra['eh_ruido'].mean():.1%})")

teste_regra = df_regra.loc[X_test.index, ['treino_recomendado', 'regra_pura', 'eh_ruido']].copy()
teste_regra['previsto_modelo'] = predicoes
teste_regra['modelo_acertou_real'] = teste_regra['previsto_modelo'] == teste_regra['treino_recomendado']
teste_regra['modelo_bateu_regra_pura'] = teste_regra['previsto_modelo'] == teste_regra['regra_pura']

n_erros = (~teste_regra['modelo_acertou_real']).sum()
n_erros_em_ruido = ((~teste_regra['modelo_acertou_real']) & teste_regra['eh_ruido']).sum()
acuracia_vs_regra_pura = teste_regra['modelo_bateu_regra_pura'].mean()

print()
print('No conjunto de teste:')
print('  erros do modelo em relacao ao rotulo real:', n_erros)
print('  desses erros, quantos caem em registros com ruido de rotulo:', n_erros_em_ruido)
print('  acuracia do modelo contra a regra pura (sem ruido):', round(acuracia_vs_regra_pura, 4))

Registros com ruido de rotulo na base toda (regra pura != rotulo real): 24 de 300 (8.0%)

No conjunto de teste:
  erros do modelo em relacao ao rotulo real: 6
  desses erros, quantos caem em registros com ruido de rotulo: 2
  acuracia do modelo contra a regra pura (sem ruido): 0.9333


**Leitura critica (com os numeros reais acima):** apenas 2 dos 6 erros do modelo caem em registros marcados como ruido de rotulo — ou seja, a maior parte dos erros (4 de 6) acontece em registros limpos, sem ruido. A hipotese de "quase todo erro e ruido" nao se confirma por completo: a arvore recupera boa parte da regra de geracao (acuracia de 93,33% contra a regra pura, acima dos 90% contra o rotulo com ruido), mas tambem comete alguns erros proprios, nao explicados so pelo ruido. A matriz de confusao da secao 8 da uma pista de onde: o Treino D e a classe mais dificil (recall de 0,78, menor entre as quatro), sendo por vezes confundido com A, B ou C — provavelmente nas fronteiras entre regras parecidas da arvore. Conclusao ajustada: o resultado de 0,90 reflete majoritariamente a recuperacao da regra sintetica (nao ha vazamento e o teto teorico do ruido explica parte do erro), mas uma fracao do erro e genuina do modelo nessas fronteiras — ponto a investigar na AP2 com validacao cruzada e analise por classe, em vez de atribuir 100% do resultado ao ruido injetado.

## 11. Dificuldades encontradas

- A base vinha com colunas categoricas em texto, entao foi necessario converter as variaveis com `get_dummies`.
- O identificador `aluno_id` nao poderia ficar entre as features.
- A principal dificuldade foi manter a estrutura do notebook organizada e garantir que a expectativa inicial aparecesse antes do treino.
- Na entrega final, a principal dificuldade foi investigar por que a acuracia ficou acima do esperado: foi preciso reabrir o script gerador (`gerar_dataset.py`) para confirmar a regra de geracao e o percentual de ruido, em vez de aceitar o numero alto sem explicacao.
- O primeiro modelo nao precisa ser perfeito; o detalhe importante e que ele rode, gere previsoes e seja bem documentado.

## 12. O que mudou em relacao ao esboco

- O modelo (Arvore de Decisao) e a divisao treino/teste (80/20, estratificada) foram mantidos - a justificativa tecnica continua valida e foi reforcada pelos resultados de interpretabilidade abaixo.
- Foram adicionadas a matriz de confusao e o F1 por classe (secao 8), que nao existiam no esboco.
- Foi adicionada a extracao das regras aprendidas pela arvore com `export_text` (secao 9), permitindo comparar o modelo com a logica do problema em vez de trata-lo como caixa-preta.
- Foi adicionada a verificacao de vazamento de dados e a explicacao quantitativa de por que a acuracia ficou proxima do teto esperado pela base sintetica (secao 10).
- A conclusao foi reescrita para refletir essa leitura critica, em vez de apenas comemorar o resultado acima do baseline.

## 13. Backlog da AP2 (o que falta ate o Encontro 11)

- [x] Treinar baseline e primeiro modelo (Atividade 9)
- [x] Matriz de confusao e F1 por classe
- [x] Verificar vazamento de dados e explicar o resultado via comparacao com a regra de geracao
- [ ] Validacao cruzada (k=5), conforme previsto no pipeline da Atividade 7
- [ ] Discutir, com os dados da secao 10, o que o resultado de 0,90 realmente significa para alunos reais (vs. recuperacao da regra sintetica) - esse deve ser o centro da apresentacao da AP2
- [ ] Preparar os slides/apresentacao da AP2

## 14. Conclusao

Este e o primeiro modelo funcional do projeto, agora com leitura critica do resultado. O baseline (0,30) e o modelo real (0,90) foram comparados, a expectativa inicial (0,60-0,80) foi confrontada com o resultado obtido, e a diferenca entre os dois foi investigada em vez de apenas celebrada: a arvore de decisao recupera a regra de geracao da base sintetica, e o erro residual e compativel com o ruido de 8% injetado de proposito. Isso mostra que o modelo aprendeu algo real sobre o problema tal como ele foi definido, mas tambem deixa claro que o proximo passo (AP2) precisa discutir os limites desse resultado diante de dados reais, alem de aplicar validacao cruzada e as metricas por classe ja levantadas aqui.